# Camera sheets and the narrative on 200 questions: a pilot

The pilot of `README.md` in this folder (`experiments/2026-11-20-camera/`): does giving Qwen3.5-2B the camera's path,
along with the pipeline's objects, help it on VSI-Bench, and in which form? It is an exploratory run, to decide what to
develop, not a result for the report. The questions are explore200's
(`experiments/2026-11-18-scene-sheet-formats/explore200_ids.txt`: 25 per VSI-Bench task, relative direction 8 easy /
8 medium / 9 hard, from 25 scenes), so the video-only predictions already on Drive can be reused. Every condition:
Qwen3.5-2B, thinking off, 16 frames (`uniform128`), lmms-eval's direct-answer prompt, greedy decoding, up to 256 new
tokens, up to 8 questions of one video per batch, one rule for reading the answer (`score_final`), and the pipeline
sheet in VGGT's relative units. The cameras are always written in the objects' format.

| Condition | Objects | Cameras | Other options |
|---|---|---|---|
| `video` | – | – | copied from Drive if it checks out |
| `obj_csv` | csv | – | |
| `cam_csv` | csv | csv table | |
| `obj_text` | sentences | – | |
| `cam_text` | sentences | compass path | |
| `narrative` | in the walk | in the walk | |
| `obj_markdown_table` | markdown table | – | |
| `cam_markdown_table` | markdown table | markdown table | |
| `cam_csv_frames` | csv | csv table | frame numbers 1–16 |
| `cam_text_turn_move` | sentences | compass path | turn, then move |
| `cam_text_frames` | sentences | compass path | frame numbers 1–16 |
| `narrative_turn_move` | in the walk | in the walk | turn, then move |
| `narrative_camera_dirs` | in the walk | in the walk | object directions in the camera's tilted axes |
| `narrative_frames` | in the walk | in the walk | frame numbers 1–16 |
| `narrative_exact` | in the walk | in the walk | exact visibility |

Unless the table says otherwise: timestamps in seconds, bridged visibility (one missed frame between two sightings
counts as seen), move then turn, object directions along the floor. The conditions run in this order, the main ones
first.

- **`video` is copied** from explore200's `ssf_explore200_results/A_video.jsonl` or, failing that, the structured-CoT
  pilot's `scot_explore_results/A_video.jsonl`, if its predictions answer these 200 questions on the same frames,
  scored by the final-answer rule, with the prompts this code builds (batch by batch, the same input length) and
  replies shorter than 256 tokens. Otherwise it runs here.
- **256 new tokens:** direct replies are a letter or a number (at most 10 tokens in the CoT pilot's `A_video`). The
  limit only stops a reply that starts to reason, and those are counted.
- **Time:** before the runs, 5 questions of one video are timed in every condition, after one warm-up question, since
  the eval's first batch also warms up the GPU (the 5 answers are kept). The notebook then prints the estimated time
  for the rest on this GPU, and above `HOURS_LIMIT` (4 h) it stops until you choose `SUBSET`, a number of questions per
  question type.
- **Summary:** scores per condition, task and question type; the camera effect in each format (`cam_<f>` −
  `obj_<f>`), the narrative against the other text forms, each option against its default and every condition against
  `video`, with bootstrap 95% intervals; example replies.

**After a crash**, run the notebook again from the top: data already in this runtime is not fetched again, a condition
whose predictions on Drive answer every question is skipped, a partial one continues from its last saved batch, and the
timing is kept (`timing.json`).

**Files on Google Drive**, in `MyDrive/mvp/`: `mvp-3d-vlm-camera.zip`, the code, made in the repository (committed or
not) with

```
REV=$(git stash create); git archive --format=zip -o mvp-3d-vlm-camera.zip ${REV:-HEAD} src experiments/2026-11-18-scene-sheet-formats requirements.txt pyproject.toml
```

the scene-sheets zip (`SHEETS_ZIP`; it must have each pipeline sheet's `cameras.csv`, which the input check checks) and,
to reuse `video`, explore200's or the CoT pilot's predictions (`ssf_explore200_results/`, `scot_explore_results/`).
VSI-Bench's `test.jsonl` and the 25 videos come from Hugging Face. Predictions go to
`MyDrive/mvp/camera_explore_results/`.

## Paths

In [ ]:
USE_DRIVE = True                     # False: put the files in /content yourself (Files pane) and set DRIVE_DIR to "/content"
DRIVE_DIR = "/content/drive/MyDrive/mvp"
CODE_ZIP = f"{DRIVE_DIR}/mvp-3d-vlm-camera.zip"  # the code (the first cell says how to make it)
DATA_SOURCE = "huggingface"          # "huggingface": VSI-Bench from Hugging Face + SHEETS_ZIP | "bundle": DATA_TAR from the cluster
SHEETS_ZIP = f"{DRIVE_DIR}/scene_sheets_75.zip"  # the scene-sheets zip: set to its name on your Drive
DATA_TAR = f"{DRIVE_DIR}/ssf_explore_data.tar.gz"  # only for DATA_SOURCE = "bundle"
WORK = "/content/mvp-3d-vlm"         # the code and data are unpacked here
RESULTS_DIR = f"{DRIVE_DIR}/camera_explore_results" if USE_DRIVE else "/content/camera_explore_results"  # this notebook's predictions (on Drive, they survive a disconnect and the runs resume)
REUSE_VIDEO = [f"{DRIVE_DIR}/ssf_explore200_results/A_video.jsonl",  # explore200's video-only predictions
               f"{DRIVE_DIR}/scot_explore_results/A_video.jsonl"]    # the structured-CoT pilot's (the first that checks out is used)

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

In [ ]:
# Every condition has the same model, frames, token limit, batches and answer rule (COMMON); they differ only in the
# scene information between the video and the question. name -> the pipeline sheet's options (None: video only).
MODEL = "Qwen/Qwen3.5-2B"            # thinking off: the eval turns it off for Qwen3.5
SSF = f"{WORK}/experiments/2026-11-18-scene-sheet-formats"  # the format experiment: the 200 questions
IDS_PATH = f"{SSF}/explore200_ids.txt"
SHEETS_ROOT = f"{WORK}/data/scene_sheets_75"
MAX_NEW_TOKENS = 256                 # direct answers are a letter or a number; this only stops a reply that reasons
BATCH_SIZE = 8                       # the eval's default, as explore200's and the CoT pilot's runs (the reuse check needs it)
TOKEN_BUDGET = 120000                # the eval's default --token-budget (input tokens per batch)
COMMON = ["--model", MODEL, "--nframes", "16", "--sampling", "uniform128", "--max-new-tokens", str(MAX_NEW_TOKENS),
          "--final-answer", "--batch-size", str(BATCH_SIZE), "--token-budget", str(TOKEN_BUDGET)]
DEFAULTS = {"cameras": False, "time_unit": "seconds", "visibility": "bridged", "step_order": "move-turn",
            "object_directions": "floor"}
CONDITIONS = {                       # in the order they run: the main conditions first, then the variants
    "video": None,
    "obj_csv": {"fmt": "csv"},
    "cam_csv": {"fmt": "csv", "cameras": True},
    "obj_text": {"fmt": "text"},
    "cam_text": {"fmt": "text", "cameras": True},                # the cameras as text: the compass path
    "narrative": {"fmt": "narrative"},                           # objects and cameras in one walk
    "obj_markdown_table": {"fmt": "markdown_table"},
    "cam_markdown_table": {"fmt": "markdown_table", "cameras": True},
    "cam_csv_frames": {"fmt": "csv", "cameras": True, "time_unit": "frames"},
    "cam_text_turn_move": {"fmt": "text", "cameras": True, "step_order": "turn-move"},
    "cam_text_frames": {"fmt": "text", "cameras": True, "time_unit": "frames"},
    "narrative_turn_move": {"fmt": "narrative", "step_order": "turn-move"},
    "narrative_camera_dirs": {"fmt": "narrative", "object_directions": "camera"},
    "narrative_frames": {"fmt": "narrative", "time_unit": "frames"},
    "narrative_exact": {"fmt": "narrative", "visibility": "exact"},
}
COMPARISONS = [                      # first minus second (README, Comparisons)
    ("cam_csv", "obj_csv"), ("cam_markdown_table", "obj_markdown_table"), ("cam_text", "obj_text"),  # do cameras help
    ("narrative", "cam_text"), ("narrative", "obj_text"), ("narrative", "obj_csv"),                   # the narrative
    ("cam_csv_frames", "cam_csv"), ("cam_text_turn_move", "cam_text"), ("cam_text_frames", "cam_text"),  # the options
    ("narrative_turn_move", "narrative"), ("narrative_camera_dirs", "narrative"), ("narrative_frames", "narrative"),
    ("narrative_exact", "narrative"),
]
REUSE = True                         # False: run video here instead of copying it
TIMING_N = 5                         # questions timed per condition before the full runs: one video, one batch
HOURS_LIMIT = 4                      # an estimate above this stops the notebook to let you choose SUBSET
SUBSET = None                        # None: all 200 if the runs fit in HOURS_LIMIT, else stop and ask;
                                     # "all": all 200 anyway; an int k: k questions per question type (Runs)


def opts(name):
    """The condition's sheet options with the defaults filled in, or None for video only."""
    o = CONDITIONS[name]
    return None if o is None else {**DEFAULTS, **o}


def family(name):
    o = opts(name)
    if o is None:
        return "video only"
    if o["fmt"] == "narrative":
        return "narrative"
    if not o["cameras"]:
        return "objects only"
    return "camera text" if o["fmt"] == "text" else "camera table"


def eval_args(name):
    """The eval's arguments for a condition, without --ids and --out. Every sheet option is given explicitly."""
    o, args = opts(name), list(COMMON)
    if o:
        args += ["--sheet-source", "pipeline", "--sheet-format", o["fmt"], "--sheets-root", SHEETS_ROOT,
                 "--time-unit", o["time_unit"], "--visibility", o["visibility"], "--step-order", o["step_order"],
                 "--object-directions", o["object_directions"]] + (["--cameras"] if o["cameras"] else [])
    return args


assert all(x in CONDITIONS and y in CONDITIONS for x, y in COMPARISONS)

## Setup

In [ ]:
import os, shutil, subprocess, sys, zipfile

os.makedirs(WORK, exist_ok=True)
with zipfile.ZipFile(CODE_ZIP) as z:
    z.extractall(WORK)
    CODE_COMMIT = z.comment.decode() or "unknown"  # git archive writes the commit it archived as the zip's comment
print(f"code: {CODE_ZIP}, commit {CODE_COMMIT}")
# what this notebook needs from the code: the camera and narrative options, reply lengths in the predictions, the
# processor's batch, the 200 questions
needs = [("src/mvp/eval/vsibench.py", "--object-directions"), ("src/mvp/eval/vsibench.py", "hit_token_limit"),
         ("src/mvp/benchmarks/scene_sheet.py", "def render_narrative"), ("src/mvp/models/qwen_vl.py", "def video_inputs"),
         ("experiments/2026-11-18-scene-sheet-formats/explore200_ids.txt", "")]
old = sorted({f for f, text in needs if not os.path.exists(f"{WORK}/{f}") or text not in open(f"{WORK}/{f}").read()})
assert not old, f"{CODE_ZIP} lacks what this notebook needs ({', '.join(old)}): make it again as the first cell says"

# requirements.txt pins torch 2.14.1+cu130; Colab's own torch is built for its driver, so it is kept unless this is False
KEEP_COLAB_TORCH = True
reqs = []
for line in open(f"{WORK}/requirements.txt").read().splitlines():
    line = line.split("#")[0].strip()
    if not line or (KEEP_COLAB_TORCH and (line.startswith("--extra-index-url") or line.split("==")[0] in ("torch", "torchvision"))):
        continue
    reqs.append(line)
# decord has no wheels for recent Python versions; eva-decord is a drop-in build imported under the same name
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *[r for r in reqs if r != "decord"]], check=True)
if subprocess.run([sys.executable, "-m", "pip", "install", "-q", "decord"]).returncode != 0:
    print("decord has no wheel for this Python; installing eva-decord instead")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "eva-decord"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", WORK], check=True)
sys.path.insert(0, f"{WORK}/src")
print("installed:", ", ".join(reqs))

In [ ]:
import torch

print("torch", torch.__version__, "CUDA", torch.version.cuda)
assert torch.cuda.is_available(), "no GPU: Runtime > Change runtime type > GPU (L4 or A100)"
GPU = torch.cuda.get_device_name(0)
cap = torch.cuda.get_device_capability(0)
print(f"GPU: {GPU}, compute capability {cap[0]}.{cap[1]}, {torch.cuda.get_device_properties(0).total_memory / 2**30:.0f} GiB")
if "T4" in GPU or cap < (8, 0):
    print("WARNING: this GPU has no native bfloat16 (T4 or older). The model runs in bfloat16, which is emulated here "
          "and several times slower; switch to an L4 or A100 if you can.")

## Data

Laid out as the code expects it: `data/raw/VSI-Bench/` (`test.jsonl`, `<dataset>/<scene>.mp4`),
`data/vggt/<dataset>/<scene>/` (`frame_indices.npy`, `meta.json`) and `data/scene_sheets_75/` (`index.csv`, the
questions' pipeline sheet folders with `objects.csv` and `cameras.csv`, each scene's `adapter_report.json`).

In [ ]:
import csv, io, json, shutil, tarfile, zipfile
from pathlib import Path

import numpy as np

VSI_REPO = "nyu-visionx/VSI-Bench"
RAW, VGGT, SHEETS = Path(WORK) / "data/raw/VSI-Bench", Path(WORK) / "data/vggt", Path(SHEETS_ROOT)
IDS = [int(x) for x in Path(IDS_PATH).read_text().split()]


def save(data, dest):
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_bytes(data)


def export_frames(video, max_frames=128):
    """frame_indices and fps exactly as the VGGT export takes them (mvp.reconstruction.extract.load_frames): decord's
    frame count and average fps, and min(128, N) frames spaced by np.linspace(0, N - 1, dtype=int)."""
    from decord import VideoReader, cpu
    vr = VideoReader(str(video), ctx=cpu(0))
    n = len(vr)
    idx = np.linspace(0, n - 1, num=min(max_frames, n), dtype=int)
    return idx, n, float(vr.get_avg_fps())


if DATA_SOURCE == "bundle":
    with tarfile.open(DATA_TAR) as t:
        t.extractall(WORK)
    print("unpacked", DATA_TAR)
else:
    from huggingface_hub import HfFileSystem, hf_hub_download

    # 1. the sheets: index.csv, each question's pipeline sheet folder (objects.csv, cameras.csv, question.json) and
    #    each scene's adapter_report.json. No gold sheets: no condition has one.
    with zipfile.ZipFile(SHEETS_ZIP) as z:
        names = z.namelist()
        index_name = min((n for n in names if n.endswith("index.csv")), key=len)  # the top-level one
        prefix = index_name[: -len("index.csv")]
        rows = {int(r["id"]): r for r in csv.DictReader(io.TextIOWrapper(z.open(index_name), encoding="utf-8"))}
        absent = [q for q in IDS if q not in rows]
        assert not absent, f"{SHEETS_ZIP}: questions {absent} are not in its index.csv"
        scenes = sorted({rows[q]["scene"] for q in IDS})
        wanted = {rows[q]["pipeline_sheet"] for q in IDS}
        wanted |= {f"{s.replace('/', '__')}/subset-uniform16/scene/adapter_report.json" for s in scenes}
        save(z.read(index_name), SHEETS / "index.csv")
        n_files = 0
        for name in names:
            parts = name[len(prefix):].split("/")
            if (name.startswith(prefix) and not name.endswith("/")
                    and any("/".join(parts[:i]) in wanted for i in range(1, len(parts) + 1))):
                save(z.read(name), SHEETS.joinpath(*parts))
                n_files += 1
        print(f"sheets: index.csv and {n_files} files: the pipeline sheets of the {len(IDS)} questions, {len(scenes)} scenes")
        # 2. frame_indices.npy and meta.json: from the zip if it has them for these scenes
        in_zip = {}
        for s in scenes:
            for f in ("frame_indices.npy", "meta.json"):
                hits = [n for n in names if n.endswith(f"/{s}/{f}") or n.endswith(f"/{s.replace('/', '__')}/{f}")]
                if hits:
                    in_zip[(s, f)] = min(hits, key=len)
        for s in scenes:
            if (s, "frame_indices.npy") in in_zip and (s, "meta.json") in in_zip:
                for f in ("frame_indices.npy", "meta.json"):
                    save(z.read(in_zip[(s, f)]), VGGT / s / f)
                print(f"{s}: frame_indices.npy and meta.json taken from the zip ({in_zip[(s, 'meta.json')]})")

    # 3. VSI-Bench's test.jsonl and the scenes' videos, read out of the per-dataset zips without downloading them whole
    RAW.mkdir(parents=True, exist_ok=True)
    if not (RAW / "test.jsonl").exists():  # after a crash in the same runtime, what is here is not fetched again
        shutil.copy(hf_hub_download(VSI_REPO, "test.jsonl", repo_type="dataset"), RAW / "test.jsonl")
    fs = HfFileSystem()
    for dataset in sorted({s.split("/")[0] for s in scenes if not (RAW / f"{s}.mp4").exists()}):
        with fs.open(f"datasets/{VSI_REPO}/{dataset}.zip", "rb", block_size=2**20) as f, zipfile.ZipFile(f) as z:
            for s in [s for s in scenes if s.startswith(dataset + "/") and not (RAW / f"{s}.mp4").exists()]:
                save(z.read(f"{s}.mp4"), RAW / f"{s}.mp4")
                print(f"{s}.mp4: {(RAW / f'{s}.mp4').stat().st_size / 2**20:.1f} MB from {dataset}.zip")

    # 4. recompute frame_indices.npy and meta.json where the zip had none
    for s in scenes:
        if (VGGT / s / "meta.json").exists() and (VGGT / s / "frame_indices.npy").exists():
            continue
        idx, n, fps = export_frames(RAW / f"{s}.mp4")
        (VGGT / s).mkdir(parents=True, exist_ok=True)
        np.save(VGGT / s / "frame_indices.npy", idx.astype(np.int32))
        (VGGT / s / "meta.json").write_text(json.dumps({
            "video": f"data/raw/VSI-Bench/{s}.mp4", "num_video_frames": n, "video_fps": fps, "num_frames": len(idx),
            "recomputed": "explore_camera.ipynb: frame_indices and video_fps as mvp.reconstruction.extract.load_frames "
                          "computes them; no VGGT outputs"}, indent=1))
        print(f"{s}: recomputed from the video: {n} frames, {fps} fps, {len(idx)} frame indices")

In [ ]:
# The video's "<t seconds>" labels come from the processor, from the fps the eval's video reader reports
# (qwen_vl_utils uses torchcodec, decord or torchvision, the first one installed). torchcodec measures the fps over the
# whole stream (frames / duration) and decord reads it from the header, and the two can differ in the last bits of the
# float (30.000000000000004 against 30.0), which flips a pair's label when its mean time is on a .x5 boundary. The sheets'
# timestamps (scene_sheet.frame_timestamps) use meta.json's video_fps, so it is set here to exactly what the eval's
# reader reports, read with the eval's own QwenVL.load_video; the export's value is kept as export_video_fps.
import inspect
import json
import sys

sys.path.insert(0, f"{WORK}/src")
from mvp.benchmarks import scene_sheet as ss
from mvp.models.qwen_vl import QwenVL
from qwen_vl_utils.vision_process import get_video_reader_backend

reader = QwenVL.__new__(QwenVL)  # load_video only needs QwenVL's pixel limits, not the model or the processor
limits = inspect.signature(QwenVL.__init__).parameters
reader.min_pixels, reader.max_pixels = limits["min_pixels"].default, limits["max_pixels"].default
backend = get_video_reader_backend()
index = ss.load_index(SHEETS)
changed = []
for s in sorted({index[q]["scene"] for q in IDS}):
    path = VGGT / s / "meta.json"
    meta = json.loads(path.read_text())
    _, video_meta, _ = reader.load_video(str(RAW / f"{s}.mp4"), nframes=ss.N_FRAMES)
    fps = float(video_meta["fps"])
    meta.setdefault("export_video_fps", meta["video_fps"])
    if meta["video_fps"] != fps or meta.get("video_reader_backend") != backend:
        if meta["video_fps"] != fps:
            changed.append(f"{s}: {meta['video_fps']!r} -> {fps!r}")
        meta.update(video_fps=fps, video_reader_backend=backend)
        path.write_text(json.dumps(meta, indent=1))
ss.frame_timestamps.cache_clear()  # timestamps computed before this cell used the old fps
print(f"video reader: {backend}; meta.json's video_fps changed to the reader's in {len(changed)} scenes"
      + "".join(f"\n  {c}" for c in changed))

In [ ]:
# Check 1: the 16 frames the eval picks (uniform128: 16 of the 128 frame_indices) are the frames the sheets were made on
# (adapter_report.json's source_video_frames). If not, the sheets describe other frames than the model sees.
import sys
sys.path.insert(0, f"{WORK}/src")
from mvp.benchmarks import scene_sheet as ss
from mvp.reconstruction.sa_sampling import uniform_subset

index = ss.load_index(SHEETS)
problems = []
for s in sorted({index[q]["scene"] for q in IDS}):
    idx = np.load(VGGT / s / "frame_indices.npy")
    report = json.loads((SHEETS / s.replace("/", "__") / "subset-uniform16" / "scene" / "adapter_report.json").read_text())
    picked = idx[uniform_subset(len(idx), 16)].tolist()
    fps = json.loads((VGGT / s / "meta.json").read_text())["video_fps"]
    print(f"{s}: {len(idx)} frame indices, fps {fps}; the eval's 16 frames {picked}")
    if report["selected_frames"] != uniform_subset(128, 16):
        problems.append(f"{s}: the sheets were made on frames {report['selected_frames']} of the 128, not uniform16")
    if picked != report["source_video_frames"]:
        problems.append(f"{s}: the eval would show video frames {picked}, but the sheets were made on "
                        f"{report['source_video_frames']}")
if problems:
    raise RuntimeError("The video frames don't match the sheets, so the sheets would describe other frames than the "
                       "model sees:\n  " + "\n  ".join(problems) +
                       "\nThe videos or the recomputed frame_indices.npy differ from the cluster's. Use the cluster's "
                       "data (DATA_SOURCE = \"bundle\") instead.")
print("OK: every scene's 16 frames are the ones its sheets were made on")

## Input check

In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, f"{WORK}/src")
from decord import VideoReader
from mvp.benchmarks import scene_sheet as ss
from mvp.benchmarks.vsibench import load_vsibench
from mvp.paths import VGGT_DIR, VSI_DIR

IDS = [int(x) for x in Path(IDS_PATH).read_text().split()]
index = ss.load_index(Path(SHEETS_ROOT))
vsi = load_vsibench("full").set_index("id")
need = [VSI_DIR / "test.jsonl", Path(SHEETS_ROOT) / "index.csv"]
for q in IDS:
    row = index[q]
    pipe = Path(SHEETS_ROOT) / row["pipeline_sheet"]
    need += [Path(vsi.loc[q, "video_path"]), VGGT_DIR / row["scene"] / "frame_indices.npy",
             VGGT_DIR / row["scene"] / "meta.json", pipe / "question.json", pipe / "objects.csv", pipe / "cameras.csv",
             pipe.parents[1] / "scene" / "adapter_report.json"]
missing = sorted({str(p) for p in need if not p.exists()})
assert not missing, "missing files:\n" + "\n".join(missing)
assert all(vsi.loc[q, "question"] == index[q]["question"] for q in IDS), "index.csv and test.jsonl disagree on a question"
for q in IDS:  # every cameras.csv has one row per frame (load_cameras raises otherwise)
    ss.load_cameras(q, Path(SHEETS_ROOT), VGGT_DIR)
n_empty = sum(not ss.load_objects(q, "pipeline", Path(SHEETS_ROOT), vggt_root=VGGT_DIR) for q in IDS)
print(f"all {len(set(map(str, need)))} files present for {len(IDS)} questions; every cameras.csv has 16 frames; "
      f"{n_empty} questions have no objects in their pipeline sheet")

for scene in sorted({index[q]["scene"] for q in IDS}):
    video = Path(vsi[vsi["scene_name"] == scene.split("/")[1]]["video_path"].iloc[0])
    vr = VideoReader(str(video))
    fps = json.loads((VGGT_DIR / scene / "meta.json").read_text())["video_fps"]
    print(f"{scene}: {len(vr)} frames, {fps:.2f} fps, {len(vr) / fps:.0f} s, "
          f"{sum(index[q]['scene'] == scene for q in IDS)} questions")

## Prompts

The conditions must differ only in the scene information between the video and the question. Check 2: the prompts
the eval builds from each condition's arguments (`--print-prompt`: the processor and the videos, not the weights), all
200 for `narrative` and the first `PRINT_N` for the others: the video's `<t seconds>` labels are the sheets'
timestamps, every time a sheet writes is one of them, and each prompt is `build_sheet_prompt`'s with the condition's
options. Check 3, for all 200 questions in every condition: the text after the video is lmms-eval's with the sheet
after its first line, `cam_<f>` is `obj_<f>` with the cameras after the objects, and the turn-then-move camera text
changes only the camera path.

In [ ]:
# Check 2: the eval's own prompts, from the arguments each condition runs with.
import os, re, subprocess
from mvp.paths import VGGT_DIR

HEADER = re.compile(r"^===== question (\d+) \((\S+)\), (\S+), video fps (\S+), frames (\[.*\])$", re.M)  # print_prompts'
LABEL = re.compile(r"<(\d+(?:\.\d+)?) seconds>")       # the processor's label before each pair of frames
FRAME_TOKENS = re.compile(re.escape(ss.VIDEO_PAD) + r" x (\d+)")
LINE_TIME = re.compile(r"^At (\d+(?:\.\d+)?) seconds:", re.M)  # a line of the camera text or the narrative
PRINT_N = 8                          # prompts printed per condition (all of them for FULL_CHECK)
FULL_CHECK = "narrative"
ROOT = Path(SHEETS_ROOT)


def sheet(q, name):
    """The scene information the condition puts in question q's prompt (None: video only)."""
    o = opts(name)
    if o is None:
        return None
    return ss.render_scene(q, "pipeline", o["fmt"], o["cameras"], ROOT, VGGT_DIR, o["time_unit"], o["visibility"],
                           o["step_order"], o["object_directions"])


def expected_prompt(q, name, frame_tokens=None):
    """build_sheet_prompt with the condition's options: the whole text input the eval should send."""
    o = opts(name)
    if o is None:
        return ss.build_sheet_prompt(q, None, None, ROOT, VGGT_DIR, frame_tokens=frame_tokens)
    return ss.build_sheet_prompt(q, "pipeline", o["fmt"], ROOT, VGGT_DIR, frame_tokens=frame_tokens, cameras=o["cameras"],
                                 time_unit=o["time_unit"], visibility=o["visibility"], step_order=o["step_order"],
                                 object_directions=o["object_directions"])


def sheet_times(q, name):
    """Every time, in seconds, the condition's sheet writes: the objects' visible ranges, the camera table's time
    column and the lines of the camera text or the narrative."""
    o = opts(name)
    times = set(LINE_TIME.findall(sheet(q, name)))
    if o["fmt"] != "narrative":
        objects = ss.load_objects(q, "pipeline", ROOT, vggt_root=VGGT_DIR, visibility=o["visibility"])
        times |= {t for obj in objects for pair in ss.ranges_of(obj["visible"]) for t in pair}
    if o["cameras"] and o["fmt"] != "text":
        times |= {c["time"] for c in ss.load_cameras(q, ROOT, VGGT_DIR)}
    return times


problems, checked = [], {}
for name in CONDITIONS:
    n = len(IDS) if name == FULL_CHECK else PRINT_N
    cmd = [sys.executable, "-m", "mvp.eval.vsibench", *eval_args(name), "--ids", IDS_PATH, "--print-prompt", str(n)]
    printed = subprocess.run(cmd, cwd=WORK, env={**os.environ, "PYTHONPATH": f"{WORK}/src"}, capture_output=True, text=True)
    assert printed.returncode == 0, f"{name}: " + printed.stdout[-3000:] + printed.stderr[-3000:]
    heads = list(HEADER.finditer(printed.stdout))
    checked[name] = len(heads)
    if len(heads) != n:
        problems.append(f"{name}: {len(heads)} prompts printed, not {n}")
    for head, nxt in zip(heads, heads[1:] + [None]):
        q = int(head.group(1))
        block = printed.stdout[head.end():nxt.start() if nxt else len(printed.stdout)]
        labels = LABEL.findall(block)
        row = index[q]
        expected = list(ss.frame_timestamps((ROOT / row["pipeline_sheet"]).parents[1] / "scene", row["scene"],
                                            VGGT_DIR)[::ss.MERGE])
        if labels != expected:
            problems.append(f"{name}, question {q}: the video's labels {labels} are not the sheet's timestamps {expected}")
        o = opts(name)
        if o and o["time_unit"] == "seconds" and not sheet_times(q, name) <= set(labels):
            problems.append(f"{name}, question {q}: times {sorted(sheet_times(q, name) - set(labels))} are not labels")
        tokens = FRAME_TOKENS.search(block)
        if block.strip("\n") != expected_prompt(q, name, tokens.group(1) if tokens else None).strip("\n"):
            problems.append(f"{name}, question {q}: the prompt is not build_sheet_prompt's with the condition's options")
if problems:
    raise RuntimeError("The eval's prompts are not as they should be:\n  " + "\n  ".join(problems[:20]))
print(f"OK: {sum(checked.values())} prompts ({', '.join(f'{k} {v}' for k, v in checked.items())}): the video's labels "
      "are the sheets' timestamps, every time a sheet writes is one of them, and each prompt is build_sheet_prompt's "
      "with the condition's options")

In [ ]:
# Check 3: what the conditions differ in, question by question, in the text the eval puts after the video (its own
# mvp.eval.vsibench.prompt_parts, with the eval's defaults for everything this notebook doesn't set).
import argparse
from mvp.benchmarks import vsibench as vb
from mvp.benchmarks.vsibench import load_vsibench
from mvp.eval import vsibench as ev

ORDER = load_vsibench("full")
ORDER = ORDER[ORDER["id"].isin(IDS)]  # the eval's order: test.jsonl's, questions grouped by video as it runs them
DOCS = {d["id"]: d for d in ORDER.to_dict("records")}
TYPE = {q: DOCS[q]["question_type"] for q in IDS}


def namespace(name):
    """The eval's arguments prompt_parts reads: this condition's sheet options, the eval's defaults otherwise."""
    o = opts(name)
    options = {k: (o or DEFAULTS)[k] for k in DEFAULTS}  # cameras, time_unit, visibility, step_order, object_directions
    return argparse.Namespace(sheets=o is not None, sheet_source="pipeline" if o else None,
                              sheet_format=o["fmt"] if o else None, sheets_root=SHEETS_ROOT, vggt_root=str(VGGT_DIR),
                              metric_scale=None, layout="sit", reasoning="direct", **options)


TEXT = {name: {q: ev.prompt_parts(DOCS[q], namespace(name)) for q in IDS} for name in CONDITIONS}
SHEET = {name: {q: sheet(q, name) for q in IDS} for name in CONDITIONS}
problems = []
for q in IDS:
    d = DOCS[q]
    for name in CONDITIONS:
        pre, post = TEXT[name][q]
        if pre:
            problems.append(f"question {q}, {name}: text before the video")
        if post != vb.build_prompt(d, SHEET[name][q]):
            problems.append(f"question {q}, {name}: not lmms-eval's prompt with the sheet after its first line")
        if post not in expected_prompt(q, name):
            problems.append(f"question {q}, {name}: the text is not build_sheet_prompt's")
    for f in ("csv", "markdown_table", "text"):
        if SHEET[f"cam_{f}"][q] != SHEET[f"obj_{f}"][q] + "\n" + ss.render_cameras(q, f, ROOT, VGGT_DIR):
            problems.append(f"question {q}: cam_{f} is not obj_{f} with the cameras after the objects")
    if SHEET["cam_text_turn_move"][q].split("\nCamera path:")[0] != SHEET["cam_text"][q].split("\nCamera path:")[0]:
        problems.append(f"question {q}: cam_text_turn_move changes more than the camera path")
if problems:
    raise RuntimeError("The conditions differ in more than the scene information:\n  " + "\n  ".join(problems[:20]))
n_empty = sum(ss.EMPTY in SHEET["obj_csv"][q] for q in IDS)
print(f"OK, for all {len(IDS)} questions in all {len(CONDITIONS)} conditions: only the scene information differs; "
      f"{n_empty} questions have no objects (their sheets say so, and still carry the cameras)")

In [ ]:
# One question's scene information in every condition, after the whole cam_csv prompt (chat template, the video's
# "<t seconds>" labels with one <|video_pad|> per frame pair, the sheet, the question). Change Q to see another.
Q = next(q for q in IDS if TYPE[q].startswith("object_rel_direction") and ss.EMPTY not in SHEET["obj_csv"][q])
print(f"question {Q} ({TYPE[Q]}): ground truth {DOCS[Q]['ground_truth']}\n")
print("===== cam_csv, the whole prompt =====")
print(expected_prompt(Q, "cam_csv"))
for name in CONDITIONS:
    if name not in ("video", "cam_csv"):
        print(f"\n===== {name}: the scene information =====")
        print(SHEET[name][Q])

## Input lengths

Every question's input length in every condition, as the eval's batches have it (the processor only, no weights). They
show what each sheet adds, give the batches for the time estimate, and decide whether the video-only predictions on
Drive were made with these prompts (Runs).

In [ ]:
import pandas as pd
from mvp.models.qwen_vl import QwenVL

PROMPTER = QwenVL(MODEL, weights=False)  # the processor and the tokenizer, without the weights


def chat_tokens(name, q):
    """The tokens of the condition's chat text for question q with the video as its one placeholder. The processor
    expands only the placeholder, between special tokens the text is split at, so a condition's input length is the
    video-only one plus this count's difference from the video-only text's."""
    pre, post = TEXT[name][q]
    text = PROMPTER.processor.apply_chat_template(PROMPTER._messages(post, "video", pre=pre), tokenize=False,
                                                  add_generation_prompt=True, **PROMPTER.template_kwargs)
    return len(PROMPTER.processor.tokenizer(text, add_special_tokens=False)["input_ids"])


def processor_lengths(name, docs, video, meta, vkw):
    """Input lengths as the eval's batch has them: the processor on the video and the prompts."""
    parts = [TEXT[name][d["id"]] for d in docs]
    mask = PROMPTER.video_inputs(video, meta, vkw, [p for _, p in parts], pres=[q for q, _ in parts])["attention_mask"]
    return {d["id"]: int(n) for d, n in zip(docs, mask.sum(1))}


LENGTHS = {name: {} for name in CONDITIONS}
VIDEO_OF, FRAMES, VIS_TOKENS = {}, {}, {}
for k, (path, group) in enumerate(ORDER.groupby("video_path", sort=False)):
    video, meta, vkw = PROMPTER.load_video_frames(path, ev.frame_indices(path, "uniform128", 16, str(VGGT_DIR)))
    VIS_TOKENS[path] = video.shape[0] * video.shape[2] * video.shape[3] // (2 * 32 * 32)  # as the eval counts them
    docs = group.to_dict("records")
    for d in docs:
        VIDEO_OF[d["id"]], FRAMES[d["id"]] = path, list(meta["frames_indices"])
    LENGTHS["video"].update(processor_lengths("video", docs, video, meta, vkw))
    for name in CONDITIONS:
        LENGTHS[name].update({d["id"]: LENGTHS["video"][d["id"]] + chat_tokens(name, d["id"]) - chat_tokens("video", d["id"])
                              for d in docs})
        if k == 0:  # on the first video, check the shortcut against the processor in every condition
            assert processor_lengths(name, docs, video, meta, vkw) == {d["id"]: LENGTHS[name][d["id"]] for d in docs}, name
lengths = pd.DataFrame(LENGTHS).loc[IDS]
SHEET_TOKENS = lengths.sub(lengths["video"], axis=0)  # what the scene information adds to the video-only prompt
print(f"input tokens per question ({len(IDS)} questions; the video is {min(VIS_TOKENS.values())} to "
      f"{max(VIS_TOKENS.values())} of them), and what the scene information adds:")
display(pd.DataFrame({"family": {n: family(n) for n in CONDITIONS}, "input mean": lengths.mean(),
                      "input max": lengths.max(), "sheet mean": SHEET_TOKENS.mean(),
                      "sheet max": SHEET_TOKENS.max()}).round(0))

## Runs

First `video`, copied from Drive if its predictions check out. Then 5 questions of one video are timed in every
condition (after a warm-up question), and the estimate for the rest decides the questions: all 200 or, with
`SUBSET = k`, k questions of every
question type (all of a type that has fewer: relative direction has 8, 8 and 9), taken scene by scene in a random
order of the scenes (seed 0), since the time grows with the number of batches. The Summary uses the questions every
condition answered.

In [ ]:
import json, math, os, subprocess, sys, threading, time
from collections import Counter
from pathlib import Path

os.makedirs(RESULTS_DIR, exist_ok=True)
BY_VIDEO = ORDER.groupby("video_path", sort=False)["id"].apply(list)  # the eval's order of videos and questions


def out(name):
    """The condition's predictions."""
    return Path(RESULTS_DIR) / f"{name}.jsonl"


def answered(path):
    """The question ids a prediction file already answers. A crash can leave a half-written last line, which would stop
    the eval's own resume (it reads every line as JSON): unreadable lines are dropped from the file first."""
    path = Path(path)
    if not path.exists():
        return set()
    lines = path.read_text().splitlines()
    good = []
    for line in lines:
        try:
            good.append(json.loads(line))
        except json.JSONDecodeError:
            pass
    if len(good) != len(lines):
        print(f"{path.name}: dropped {len(lines) - len(good)} unreadable line(s) left by a crash")
        path.write_text("".join(json.dumps(r) + "\n" for r in good))
    return {r["id"] for r in good}


def count(path, ids):
    """How many of ids a prediction file answers, reading it as it is written (for the progress lines)."""
    try:
        lines = Path(path).read_text().splitlines()
    except FileNotFoundError:
        return 0
    got = set()
    for line in lines:
        try:
            got.add(json.loads(line)["id"])
        except (json.JSONDecodeError, KeyError):
            pass
    return len(got & set(ids))


def run_eval(name, ids_path, out_path, every=300, watch=None):
    """The eval for one condition on the questions of ids_path (it skips those out_path already answers), with its
    output and, every `every` seconds, how many are answered. watch, a list: (seconds since the start, questions
    answered) is appended to it each time out_path gains answers (the eval writes them batch by batch). Returns
    (seconds, the eval's output)."""
    ids = [int(x) for x in Path(ids_path).read_text().split()]
    cmd = [sys.executable, "-m", "mvp.eval.vsibench", *eval_args(name), "--ids", str(ids_path), "--out", str(out_path)]
    print(f"===== {name}: {' '.join(cmd[1:])}", flush=True)
    t0, stop = time.time(), threading.Event()

    def progress():
        while not stop.wait(every):
            print(f"  [{name}: {count(out_path, ids)} of {len(ids)} answered, {(time.time() - t0) / 60:.0f} min]", flush=True)

    def watcher():
        last = count(out_path, ids)
        while not stop.wait(0.2):
            n = count(out_path, ids)
            if n != last:
                watch.append((time.time() - t0, n))
                last = n

    threading.Thread(target=progress, daemon=True).start()
    if watch is not None:
        threading.Thread(target=watcher, daemon=True).start()
    proc = subprocess.Popen(cmd, cwd=WORK, env={**os.environ, "PYTHONPATH": f"{WORK}/src"}, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True)
    lines = []
    for line in proc.stdout:
        print(line, end="")
        lines.append(line)
    stop.set()
    assert proc.wait() == 0, f"{name} failed"
    print(f"===== {name} done in {time.time() - t0:.0f} s\n", flush=True)
    return time.time() - t0, "".join(lines)


def batch_size(video):
    """The eval's batch size for a video: min(BATCH_SIZE, TOKEN_BUDGET // (visual tokens + 200))."""
    return max(1, min(BATCH_SIZE, TOKEN_BUDGET // (VIS_TOKENS[video] + 200)))


def batches(name, ids):
    """The batches the eval will make of the questions of ids that `name` has not answered, in its order."""
    done, ids = answered(out(name)), set(ids)
    result = []
    for video, qs in BY_VIDEO.items():
        left, bs = [q for q in qs if q in ids and q not in done], batch_size(video)
        result += [left[i:i + bs] for i in range(0, len(left), bs)]
    return result


def padded(name, batch):
    """A batch's input tokens as the model gets them: left-padded to its longest prompt."""
    return max(LENGTHS[name][q] for q in batch) * len(batch)

In [ ]:
# video: copied from Drive when the predictions answer the same questions, on the same frames, scored by the
# final-answer rule, with replies shorter than the token limit, and with the prompts and batches this code makes: the
# eval records each batch's padded input length (n_input_tokens), so every batch this notebook would make must have one
# recorded length there, its longest prompt's as built here. Copied predictions are listed in reused.json.
from mvp.models.qwen_vl import SYSTEM_PROMPT


def reuse_problems(path):
    """Why the predictions in `path` cannot stand in for video (empty: they can)."""
    path = Path(path)
    if not path.exists():
        return [f"no {path}"]
    run = pd.read_json(path, lines=True)
    if len(run) != len(IDS) or set(run["id"]) != set(IDS):
        return [f"{run['id'].nunique()} questions in {len(run)} lines, not these {len(IDS)}"]
    problems = []
    summary = path.with_name(path.stem + "_summary.json")
    if summary.exists():  # the run's settings, where its summary has them
        s = json.loads(summary.read_text())
        want = {"model": MODEL, "nframes": 16, "sampling": "uniform128", "layout": "sit", "system": SYSTEM_PROMPT,
                "final_answer": True, "reasoning": "direct", "scene_sheet": None}
        problems += [f"its summary has {k} = {s[k]!r}, not {v!r}" for k, v in want.items() if k in s and s[k] != v]
    if "score_first" not in run:
        problems.append("not scored with --final-answer")
    other = [q for q, f in zip(run["id"], run["frame_indices"]) if list(f) != FRAMES[q]]
    if other:
        problems.append(f"{len(other)} questions on other frames than the eval picks here")
    recorded = dict(zip(run["id"], run["n_input_tokens"]))
    bad = []
    for video, qs in BY_VIDEO.items():  # the batches this notebook makes: each must have been one batch there
        bs = batch_size(video)
        for b in (qs[i:i + bs] for i in range(0, len(qs), bs)):
            there, here = {recorded[q] for q in b}, max(LENGTHS["video"][q] for q in b)
            if there != {here}:
                bad.append(f"questions {b}: {sorted(there)} input tokens there, {here} here")
    if bad:
        problems.append(f"{len(bad)} batches had other prompts or other batches, e.g. " + "; ".join(bad[:2]))
    longest_reply = max(map(len, PROMPTER.processor.tokenizer(list(run["prediction"].astype(str)),
                                                               add_special_tokens=False)["input_ids"]))
    if longest_reply >= MAX_NEW_TOKENS:
        problems.append(f"a reply of {longest_reply} tokens, which the limit of {MAX_NEW_TOKENS} would have cut off")
    return problems


REUSED_JSON = Path(RESULTS_DIR) / "reused.json"
reused = json.loads(REUSED_JSON.read_text()) if REUSED_JSON.exists() else {}
have = answered(out("video")) & set(IDS)
if len(have) == len(IDS):
    print(f"video: all {len(IDS)} answered" + (f", copied from {reused['video']}" if "video" in reused else ""))
elif have:
    print(f"video: {len(have)} of {len(IDS)} answered here; the rest run with the others")
elif not REUSE:
    print("video: REUSE is False; it runs with the others")
else:
    for src in REUSE_VIDEO:
        problems = reuse_problems(src)
        print(f"video from {src}: " + ("; ".join(problems) if problems else "same questions, frames, prompts and rule"))
        if not problems:
            shutil.copy(src, out("video"))
            reused["video"] = src
            break
    else:
        print("video: none of REUSE_VIDEO checks out; it runs with the others")
    REUSED_JSON.write_text(json.dumps(reused, indent=1))

In [ ]:
# Time TIMING_N questions per condition: the questions of one video, the video with exactly TIMING_N questions and the
# most question types (the first in the eval's order among equals), so that the batch is the one the full run would
# make. The eval's first batch also pays for warming up the GPU, which the full run pays once, so each timing run starts
# with one question of the first video (WARM_UP) and ends with one of the last (TAIL, so that the timed batch is
# never the eval's last write); the timed batch's time is from the warm-up's answers being written to its own. They run
# into a scratch file; the timed answers are then added to the condition's predictions, so the full run doesn't redo
# them (the warm-up and tail answers are dropped: the full run answers them in their own batches), and timing.json
# keeps the times, so a resumed notebook doesn't time again on the same GPU.
WARM_UP, TAIL = BY_VIDEO.iloc[0][0], BY_VIDEO.iloc[-1][0]
inner = list(BY_VIDEO.index[1:-1])
candidates = ([v for v in inner if len(BY_VIDEO[v]) == TIMING_N] or [v for v in inner if len(BY_VIDEO[v]) > TIMING_N]
              or inner)
TIMING_VIDEO = max(candidates, key=lambda v: len({TYPE[q] for q in BY_VIDEO[v][:TIMING_N]}))
TIMING_IDS = BY_VIDEO[TIMING_VIDEO][:TIMING_N]
TIMING_IDS_PATH = Path(RESULTS_DIR) / "timing" / "timing_ids.txt"
TIMING_IDS_PATH.parent.mkdir(exist_ok=True)
TIMING_IDS_PATH.write_text("".join(f"{q}\n" for q in [WARM_UP, *TIMING_IDS, TAIL]))
print(f"timed: {TIMING_VIDEO.split('VSI-Bench/')[-1]}, questions {TIMING_IDS} ({', '.join(TYPE[q] for q in TIMING_IDS)}), "
      f"after warm-up question {WARM_UP} and before tail question {TAIL}")

TIMING_JSON = Path(RESULTS_DIR) / "timing.json"
timing = json.loads(TIMING_JSON.read_text()) if TIMING_JSON.exists() else {}
SETUP = {"gpu": GPU, "max_new_tokens": MAX_NEW_TOKENS, "batch_size": BATCH_SIZE, "questions": TIMING_IDS}
for name in CONDITIONS:
    if answered(out(name)) >= set(IDS):
        print(f"{name}: all {len(IDS)} questions answered, nothing to time")
        continue
    if "batch_s" in timing.get(name, {}) and all(timing[name].get(k) == v for k, v in SETUP.items()):
        continue  # timed before, with these settings on this GPU
    scratch = Path(RESULTS_DIR) / "timing" / f"{name}.jsonl"
    scratch.unlink(missing_ok=True)
    watch = []
    wall, _ = run_eval(name, TIMING_IDS_PATH, scratch, watch=watch)
    when = {n: t for t, n in reversed(watch)}  # questions answered -> the first time the file had that many
    n_timed = 1 + len(TIMING_IDS)
    assert 1 in when and n_timed in when, f"the watcher missed a batch ({watch}): run this cell again"
    recs = {r["id"]: r for r in map(json.loads, scratch.read_text().splitlines())}
    bs = batch_size(TIMING_VIDEO)  # one batch unless the batch size is under TIMING_N
    timing[name] = {**SETUP, "batch_s": round(when[n_timed] - when[1], 1), "start_s": round(when[1], 1),
                    "wall_s": round(wall, 1),
                    "reply_tokens": [recs[q]["n_new_tokens"] for q in TIMING_IDS],
                    "hit_token_limit": [recs[q]["hit_token_limit"] for q in TIMING_IDS],
                    "padded_tokens": sum(recs[b[0]]["n_input_tokens"] * len(b)  # the eval records the padded length
                                         for b in (TIMING_IDS[i:i + bs] for i in range(0, len(TIMING_IDS), bs)))}
    have = answered(out(name))
    with open(out(name), "a") as f:
        f.writelines(json.dumps(recs[q]) + "\n" for q in TIMING_IDS if q not in have)
    TIMING_JSON.write_text(json.dumps(timing, indent=1))
for name, t in timing.items():
    print(f"{name}: {len(t['questions'])} questions, {t['padded_tokens']:,} input tokens, in {t['batch_s']:.1f} s on "
          f"{t['gpu']} ({t['start_s']:.0f} s to start the eval and warm up); replies of {t['reply_tokens']} tokens, "
          f"{sum(t['hit_token_limit'])} cut off at {t['max_new_tokens']}")

In [ ]:
# The estimate per condition: direct answers are a few tokens, so a batch's time is mostly reading its input, and each
# batch left takes the timed batch's time scaled by its input tokens (padded, as the model gets them). The timed time
# includes loading the video, which the full run does once per video, so the estimate is a little high. Each run also
# starts the eval and warms up once (start_s).
import random


def per_type(k, seed=0):
    """k questions of every question type (all of a type that has fewer), taken scene by scene in a random order of
    the scenes, so that they come from few videos."""
    videos = list(BY_VIDEO.index)
    random.Random(seed).shuffle(videos)
    need = Counter({t: min(k, n) for t, n in Counter(TYPE.values()).items()})
    chosen = []
    for v in videos:
        for q in BY_VIDEO[v]:
            if need[TYPE[q]] > 0:
                chosen.append(q)
                need[TYPE[q]] -= 1
        if not +need:
            break
    return sorted(chosen)


def estimate(name, ids):
    """Expected seconds for `name` on the questions of ids it has not answered."""
    left = batches(name, ids)
    if not left:
        return 0.0
    t = timing[name]
    return t["start_s"] + t["batch_s"] / t["padded_tokens"] * sum(padded(name, b) for b in left)


if SUBSET not in (None, "all") and not isinstance(SUBSET, int):
    raise ValueError(f"SUBSET must be None, 'all' or a number of questions per question type, not {SUBSET!r}")
missing = [n for n in CONDITIONS if batches(n, IDS) and n not in timing]
assert not missing, f"no timing for {missing}: run the cell above"
choices = {"all": IDS, **{k: per_type(k) for k in sorted({10, 5} | ({SUBSET} if isinstance(SUBSET, int) else set()), reverse=True)}}
est = pd.DataFrame({f"SUBSET = {key!r}" if key != "all" else "all": {n: estimate(n, ids) / 60 for n in CONDITIONS}
                    for key, ids in choices.items()})
total = est.sum() / 60
print(f"Estimated minutes left per condition on this GPU ({GPU}), and in all:")
display(pd.concat([est.round(0), pd.DataFrame([total.round(1)], index=["all conditions (h)"])]))
print("questions:", {key: len(ids) for key, ids in choices.items()})
left_all = total.iloc[0]
if SUBSET is None and left_all > HOURS_LIMIT:
    raise RuntimeError(
        f"The runs on all {len(IDS)} questions would take about {left_all:.1f} h more on this GPU, over HOURS_LIMIT = "
        f"{HOURS_LIMIT} h. Choose in the settings cell (Paths): SUBSET = 10 or SUBSET = 5 (questions per question type; "
        f"{', '.join(f'{k}: {total.iloc[i]:.1f} h' for i, k in enumerate(list(choices)[1:], 1))}), or SUBSET = \"all\" "
        "to run all of them anyway; then run the notebook again from the top. Nothing done so far is redone, and the "
        "timing is kept.")
CHOICE = "all" if SUBSET in (None, "all") else SUBSET
RUN_IDS = choices[CHOICE]
RUN_IDS_PATH = Path(RESULTS_DIR) / "run_ids.txt"
RUN_IDS_PATH.write_text("".join(f"{q}\n" for q in RUN_IDS))
left = sum(estimate(n, RUN_IDS) for n in CONDITIONS)
print(f"The runs: {len(RUN_IDS)} questions (SUBSET = {CHOICE!r}), about {left / 3600:.1f} h left; per question type: "
      f"{dict(Counter(TYPE[q] for q in RUN_IDS))}")

In [ ]:
t_runs = time.time()
for name in CONDITIONS:
    done = answered(out(name)) & set(RUN_IDS)
    if len(done) == len(RUN_IDS):
        print(f"===== {name}: all {len(RUN_IDS)} questions answered, skipped", flush=True)
        continue
    if done:
        print(f"===== {name}: resuming, {len(done)} of {len(RUN_IDS)} answered", flush=True)
    run_eval(name, RUN_IDS_PATH, out(name))
    rest = sum(estimate(n, RUN_IDS) for n in CONDITIONS)
    print(f"{(time.time() - t_runs) / 60:.0f} min since the runs started; about {rest / 60:.0f} min left\n", flush=True)

## Answers: one rule, parse rate, reply lengths

Every reply is read by one rule, the eval's final-answer rule (`score_final`), applied again here to the copied
predictions too: for multiple choice, the letter of the last "Answer: X" or "answer is X", else the last letter that
stands alone in the reply's last line; for numbers, the last number in the reply. A reply it reads nothing from scores
0. Direct replies are a letter or a number, so the rule's main job is to read the rare reply that talks; "over 16
tokens" counts those, and "hit 256" the ones cut off.

In [ ]:
import random
import re

import numpy as np
from mvp.benchmarks.vsibench import (MCA_QUESTION_TYPES, METRIC_TASKS, TASKS, final_letter, final_number, parsed,
                                     score_final, summarize, task_of)

RUN_IDS = globals().get("RUN_IDS", IDS)


def read(d, reply):
    """The answer the rule reads from a reply: a letter, a number, or None."""
    return final_letter(reply) if d["question_type"] in MCA_QUESTION_TYPES else final_number(reply)


runs = {}
for name in CONDITIONS:
    if not out(name).exists():
        continue
    run = pd.read_json(out(name), lines=True).drop_duplicates("id").set_index("id")
    run = run.loc[[q for q in RUN_IDS if q in run.index]].copy()
    run["prediction"] = run["prediction"].astype(str)
    docs = [DOCS[q] for q in run.index]
    run["rescored"] = [score_final(d, p) for d, p in zip(docs, run["prediction"])]
    run["read"] = pd.Series([read(d, p) for d, p in zip(docs, run["prediction"])], index=run.index, dtype=object)
    run["parsed"] = [parsed(d, p, final=True) for d, p in zip(docs, run["prediction"])]
    # reply lengths: the eval's count; predictions made before the eval recorded it are counted with the tokenizer
    for col in ("n_new_tokens", "hit_token_limit"):
        if col not in run:
            run[col] = None
    missing = run["n_new_tokens"].isna()
    if missing.any():
        counted = [len(t) for t in PROMPTER.processor.tokenizer(list(run.loc[missing, "prediction"]),
                                                                add_special_tokens=False)["input_ids"]]
        run.loc[missing, "n_new_tokens"] = counted
        run.loc[missing, "hit_token_limit"] = [n >= MAX_NEW_TOKENS for n in counted]
    run["n_new_tokens"] = run["n_new_tokens"].astype(int)
    run["hit_token_limit"] = run["hit_token_limit"].astype(bool)
    runs[name] = run
COMMON_IDS = [q for q in RUN_IDS if all(q in r.index for r in runs.values())]
print(f"conditions with predictions: {', '.join(runs)}; {len(COMMON_IDS)} of the {len(RUN_IDS)} questions answered in "
      f"all of them" + ("" if set(runs) == set(CONDITIONS) else f"; missing: {', '.join(n for n in CONDITIONS if n not in runs)}"))
differ = {n: int((abs(r["rescored"] - r["score"]) > 1e-9).sum()) for n, r in runs.items()}
print("scores here that differ from the eval's own (the same rule, so none should):", differ)

answers = pd.DataFrame.from_dict({name: {
    "family": family(name),
    "replies": len(run),
    "parsed %": 100 * run["parsed"].mean(),
    "median tokens": run["n_new_tokens"].median(),
    "longest": run["n_new_tokens"].max(),
    "over 16 tokens": int((run["n_new_tokens"] > 16).sum()),
    f"hit {MAX_NEW_TOKENS}": int(run["hit_token_limit"].sum()),
    "input tokens, mean": lengths.loc[run.index, name].mean(),
    "mean score": 100 * run["rescored"].mean(),
} for name, run in runs.items()}, orient="index")
display(answers.round(1))

## Per-question table

In [ ]:
# Each condition's answer as the rule reads it, ✓/✗ (multiple choice) or the score (numbers), ✂ if cut off at the limit.
def cell(run, q):
    if q not in run.index:
        return "—"
    a, s = run.loc[q, "read"], run.loc[q, "rescored"]
    mark = " ✂" if run.loc[q, "hit_token_limit"] else ""
    if TYPE[q] in MCA_QUESTION_TYPES:
        return f"{a} {'✓' if s == 1 else '✗'}{mark}"
    return f"{'?' if pd.isna(a) else f'{a:g}'} ({s:.2f}){mark}"


table = pd.DataFrame({"scene": {q: DOCS[q]["dataset"] + "/" + DOCS[q]["scene_name"] for q in RUN_IDS},
                      "type": {q: TYPE[q] for q in RUN_IDS}, "truth": {q: DOCS[q]["ground_truth"] for q in RUN_IDS}})
for name, run in runs.items():
    table[name] = [cell(run, q) for q in table.index]
table.index.name = "id"
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 250)
display(table)

Full replies, for any question: `runs['narrative'].loc[QUESTION_ID, 'prediction']`.

## Summary

Score ×100 on the questions every condition answered. **Overall** is VSI-Bench's (the mean of the 8 task scores,
relative direction's three types as one task). **Direction and route** is the mean of relative direction and route
planning, the tasks the camera's path is most likely to help. **Without metric tasks** leaves out absolute distance,
object size and room size, which the sheets' relative units lower in every sheet condition alike. Sheet tokens: what
the scene information adds to the video-only prompt, on average.

In [ ]:
TYPE_ORDER = [t for k in TASKS for t in ([f"object_rel_direction_{x}" for x in ("easy", "medium", "hard")]
                                         if k == "object_rel_direction" else [k])]
GROUPS = {"overall": set(TASKS), "direction and route": {"object_rel_direction", "route_planning"},
          "without metric tasks": set(TASKS) - set(METRIC_TASKS)}
S = pd.DataFrame({n: r.loc[COMMON_IDS, "rescored"] for n, r in runs.items()})
types = pd.Series({q: TYPE[q] for q in COMMON_IDS})
qtypes = [t for t in TYPE_ORDER if t in set(types)]


def group_score(name, tasks):
    keep = types.index[types.map(task_of).isin(tasks)]
    return summarize(pd.DataFrame({"question_type": types[keep], "score": S.loc[keep, name]}))["overall"]


scores = pd.DataFrame({n: {"family": family(n), **{g: group_score(n, ts) for g, ts in GROUPS.items()},
                           **{k: v for k, v in summarize(pd.DataFrame({"question_type": types, "score": S[n]})).items()
                              if k != "overall"},
                           "sheet tokens": SHEET_TOKENS.loc[COMMON_IDS, n].mean()} for n in S}).T
for col in scores.columns[1:]:
    scores[col] = scores[col].astype(float)
print(f"{len(COMMON_IDS)} questions; per task: {dict(Counter(types.map(task_of)))}")
display(scores.round(1))

by_type = (100 * S.groupby(types).mean()).reindex(qtypes)
by_type.insert(0, "n", types.value_counts().reindex(qtypes).astype(int))
display(by_type.round(1))

## Bootstrap intervals

Each comparison as the difference and its 95% interval from 10,000 paired resamples: questions are drawn with
replacement within each question type, so every type keeps its number of questions, and all conditions get the same
draw. `*`: the interval excludes 0. **lead**: the README's reading rule, a difference of at least 5 points on direction
and route or 2 on the overall, with an interval that excludes 0. With about 25 comparisons, one or two will exclude 0
by chance, and with 25 questions a type (fewer with `SUBSET`) a question type's interval is wide: a lead is something
to confirm on more questions, not a finding.

In [ ]:
B = 10_000
rng = np.random.default_rng(0)
conds = list(S.columns)
X = S.to_numpy(float)
members = [np.flatnonzero(types.to_numpy() == t) for t in qtypes]
LEAD = {"overall": 2, "direction and route": 5}


def task_weights(tasks):
    """w such that w @ (per question type means) is the mean of the scores of the tasks in `tasks`."""
    kept = [k for k in dict.fromkeys(task_of(t) for t in qtypes) if k in tasks]
    return np.mean([[(task_of(t) == k) / sum(task_of(u) == k for u in qtypes) for t in qtypes] for k in kept], axis=0)


W = {g: task_weights(ts) for g, ts in GROUPS.items()}
point = np.stack([X[m].mean(0) for m in members])  # question type x condition
assert np.allclose(100 * W["overall"] @ point, scores.loc[conds, "overall"].to_numpy(float))
boot = np.stack([X[m][rng.integers(0, len(m), size=(B, len(m)))].mean(1) for m in members], axis=1)  # draw x type x condition


def difference(x, y):
    """{row: (difference, its bootstrap draws)} for x − y: the groups, then each question type."""
    i, j = conds.index(x), conds.index(y)
    d_point, d_boot = 100 * (point[:, i] - point[:, j]), 100 * (boot[:, :, i] - boot[:, :, j])
    return {**{g: (d_point @ w, d_boot @ w) for g, w in W.items()},
            **{t: (d_point[k], d_boot[:, k]) for k, t in enumerate(qtypes)}}


def interval(p, b):
    lo, hi = np.percentile(b, [2.5, 97.5])
    return f"{p:+.1f} [{lo:+.1f}, {hi:+.1f}]" + (" *" if lo > 0 or hi < 0 else "")


def lead(rows):
    hits = []
    for g, threshold in LEAD.items():
        p, b = rows[g]
        lo, hi = np.percentile(b, [2.5, 97.5])
        if abs(p) >= threshold and (lo > 0 or hi < 0):
            hits.append(f"{g} {'+' if p > 0 else '−'}")
    return ", ".join(hits)


pairs = [(x, y) for x, y in COMPARISONS if x in conds and y in conds]
skipped = [f"{x} − {y}" for x, y in COMPARISONS if (x, y) not in pairs]
if skipped:
    print("without predictions for both, not compared:", ", ".join(skipped))
DIFFS = {f"{x} − {y}": difference(x, y) for x, y in pairs}
VS_VIDEO = {f"{x} − video": difference(x, "video") for x in conds if x != "video"} if "video" in conds else {}
diff_numbers = [{"comparison": c, "rows": r, "difference": p, "95% low": np.percentile(b, 2.5),
                 "95% high": np.percentile(b, 97.5)} for c, rows in {**DIFFS, **VS_VIDEO}.items() for r, (p, b) in rows.items()]
print(f"{len(COMMON_IDS)} questions answered in every condition; {B:,} stratified paired resamples")


def summary_table(diffs):
    return pd.DataFrame({c: {**{g: interval(*rows[g]) for g in GROUPS}, "lead": lead(rows)} for c, rows in diffs.items()}).T


print("The comparisons (README):")
display(summary_table(DIFFS))
print("Every condition against video only:")
display(summary_table(VS_VIDEO))
print("The comparisons per question type:")
per_type_diffs = pd.DataFrame({c: {t: interval(*rows[t]) for t in qtypes} for c, rows in DIFFS.items()})
per_type_diffs.insert(0, "n", by_type["n"])
display(per_type_diffs)

## Example replies

For each family's main comparison, one question the first condition gets right and the second wrong, and one the
other way round, drawn at random (`SEED`), from relative direction and route planning when there are any. Each shows
both replies and the first condition's scene information (shortened). Then the longest replies of any condition.

In [ ]:
SEED = 0
EXAMPLE_PAIRS = [("cam_csv", "obj_csv"), ("cam_markdown_table", "obj_markdown_table"), ("cam_text", "obj_text"),
                 ("narrative", "cam_text"), ("narrative", "obj_text")]
CAMERA_TYPES = {t for t in qtypes if task_of(t) in GROUPS["direction and route"]}
rng_examples = random.Random(SEED)


def shorten(text, limit=4000):
    if len(text) <= limit:
        return text
    return text[:2 * limit // 3] + f"\n\n[... {len(text) - limit} characters left out ...]\n\n" + text[-limit // 3:]


def show_question(q):
    d = DOCS[q]
    print(f"question {q} ({TYPE[q]}, {d['dataset']}/{d['scene_name']}): {d['question']}")
    if TYPE[q] in MCA_QUESTION_TYPES:
        print("\n".join(d["options"]))
    print(f"ground truth: {d['ground_truth']}")


for x, y in EXAMPLE_PAIRS:
    if x not in S or y not in S:
        print(f"{x} / {y}: no predictions for both")
        continue
    for better, worse in ((x, y), (y, x)):
        gap = S[better] - S[worse]
        pool = sorted(q for q in COMMON_IDS if gap[q] >= 0.5)  # right in one and wrong in the other, or 0.5 apart
        pool = [q for q in pool if TYPE[q] in CAMERA_TYPES] or pool
        print("=" * 100)
        if not pool:
            print(f"{better} better than {worse}: no such question")
            continue
        q = rng_examples.choice(pool)
        print(f"{better.upper()} BETTER THAN {worse.upper()} ({len(pool)} such questions):")
        show_question(q)
        for name in (x, y):
            r = runs[name].loc[q]
            print(f"  {name}: reply {shorten(r['prediction'], 600)!r}, read {r['read']!r}, score {r['rescored']:.2f}, "
                  f"{r['n_new_tokens']} tokens")
        print(f"-" * 30, f"{x}'s scene information")
        print(shorten(SHEET[x][q]))

print("=" * 100)
longest = sorted(((r.loc[q, "n_new_tokens"], n, q) for n, r in runs.items() for q in COMMON_IDS), reverse=True)[:3]
for tokens, name, q in longest:
    if tokens <= 16:
        print(f"no reply over 16 tokens beyond these")
        break
    r = runs[name].loc[q]
    print(f"LONGEST: {name}, {tokens} tokens" + (", cut off" if r["hit_token_limit"] else ""))
    show_question(q)
    print(f"read {r['read']!r}, score {r['rescored']:.2f}")
    print("-" * 30, "reply")
    print(shorten(r["prediction"], 3000))
    print("=" * 100)

## Save

In [ ]:
# The tables next to the predictions (on Drive when USE_DRIVE), and what this run was.
table.to_csv(f"{RESULTS_DIR}/per_question.csv")
answers.to_csv(f"{RESULTS_DIR}/answers.csv")
scores.to_csv(f"{RESULTS_DIR}/scores.csv")
by_type.to_csv(f"{RESULTS_DIR}/by_question_type.csv")
lengths.to_csv(f"{RESULTS_DIR}/input_tokens.csv")
pd.DataFrame(diff_numbers).to_csv(f"{RESULTS_DIR}/differences_bootstrap.csv", index=False)
Path(f"{RESULTS_DIR}/run_info.json").write_text(json.dumps({
    "code_zip": CODE_ZIP, "code_commit": CODE_COMMIT, "gpu": GPU, "model": MODEL, "common": COMMON,
    "conditions": {n: opts(n) for n in CONDITIONS}, "eval_args": {n: eval_args(n) for n in CONDITIONS},
    "subset": globals().get("CHOICE"), "questions": len(COMMON_IDS), "reused": reused, "timing": timing}, indent=1))
print("saved to", RESULTS_DIR)